In [16]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# 1. Geometry Parameters (in mm)
# ==========================================
a = 237.0
b = 118.0
c_hor_0 = 247.0
c_vert = 59.0
d = 80.0

# ==========================================
# 2. Travel and Force Parameters
# ==========================================
travel_range = 100.0 # 100mm axis
travel_min_pct = 0.34
travel_max_pct = 0.58

x_start = travel_range * travel_min_pct
x_end = travel_range * travel_max_pct

# Array for sled position (x) in mm
x = np.linspace(x_start, x_end, 100)

# Forces: Preload 2kg, Max Force 5kg
force_min_kg = 2.0
force_max_kg = 5.0
g = 9.81 # Gravity m/s^2

# Desired pedal force (F_p) in Newton
F_p = np.linspace(force_min_kg * g, force_max_kg * g, len(x))

# ==========================================
# 3. Kinematics & Dynamics Calculations
# ==========================================
c_hor = c_hor_0 + x
c = np.sqrt(c_vert**2 + c_hor**2)

# Pedal angles
alpha = np.arccos((b**2 + c**2 - a**2) / (2 * b * c))
alpha_plus = np.arctan(c_vert / c_hor)

phi_rad = alpha + alpha_plus
phi_deg = np.degrees(phi_rad)

# Horizontal Position of Point D
D_hor = (b + d) * np.cos(phi_rad)

# Torque in Nm
lever_arm_m = (b + d) / 1000.0
T_pedal = F_p * lever_arm_m

# Loadcell Force (F_l)
gamma = np.arccos((a**2 + b**2 - c**2) / (2 * a * b))
F_l = F_p * (b + d) / (b * np.sin(gamma))

# Horizontal Pedal Force (F_px)
F_px = F_p * np.sin(phi_rad)

# ==========================================
# 4. Plotly Subplots
# ==========================================
# Wir erhöhen das vertical_spacing, um Platz für die Legenden zu schaffen
fig = make_subplots(
    rows=3, cols=2,
    specs=[
        [{"secondary_y": True}, {"secondary_y": True}],
        [{"secondary_y": True}, {"secondary_y": True}],
        [{"secondary_y": True}, {"secondary_y": False}]
    ],
    subplot_titles=(
        "1. Force vs. Sled Position",
        "2. Force vs. Horizontal Pedal Position",
        "3. Force vs. Pedal Angle",
        "4. Torque vs. Horizontal Pedal Position",
        "5. Torque vs. Pedal Angle",
        ""
    ),
    vertical_spacing=0.16,
    horizontal_spacing=0.1
)

# Styling
style_Fl = dict(color='rgba(100, 100, 100, 0.8)', width=2, dash='dash')
name_Fl = 'Loadcell F_l'

style_Fpx = dict(color='#17becf', width=2.5, dash='dot')
name_Fpx = 'Horizontal F_px'

name_Fp = 'Pedal Force F_p'
name_T = 'Torque T_pedal'

# --- Plot 1: Force vs x (Legend 1) ---
fig.add_trace(go.Scatter(x=x, y=F_p, mode='lines', name=name_Fp, line=dict(color='#1f77b4', width=3), legend='legend'), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=x, y=F_px, mode='lines', name=name_Fpx, line=style_Fpx, legend='legend'), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=x, y=F_l, mode='lines', name=name_Fl, line=style_Fl, legend='legend'), row=1, col=1, secondary_y=True)

# --- Plot 2: Force vs D_hor (Legend 2) ---
fig.add_trace(go.Scatter(x=D_hor, y=F_p, mode='lines', name=name_Fp, line=dict(color='#ff7f0e', width=3), legend='legend2'), row=1, col=2, secondary_y=False)
fig.add_trace(go.Scatter(x=D_hor, y=F_px, mode='lines', name=name_Fpx, line=style_Fpx, legend='legend2'), row=1, col=2, secondary_y=False)
fig.add_trace(go.Scatter(x=D_hor, y=F_l, mode='lines', name=name_Fl, line=style_Fl, legend='legend2'), row=1, col=2, secondary_y=True)

# --- Plot 3: Force vs phi (Legend 3) ---
fig.add_trace(go.Scatter(x=phi_deg, y=F_p, mode='lines', name=name_Fp, line=dict(color='#2ca02c', width=3), legend='legend3'), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=phi_deg, y=F_px, mode='lines', name=name_Fpx, line=style_Fpx, legend='legend3'), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=phi_deg, y=F_l, mode='lines', name=name_Fl, line=style_Fl, legend='legend3'), row=2, col=1, secondary_y=True)

# --- Plot 4: Torque vs D_hor (Legend 4) ---
fig.add_trace(go.Scatter(x=D_hor, y=T_pedal, mode='lines', name=name_T, line=dict(color='#d62728', width=3), legend='legend4'), row=2, col=2, secondary_y=False)
fig.add_trace(go.Scatter(x=D_hor, y=F_l, mode='lines', name=name_Fl, line=style_Fl, legend='legend4'), row=2, col=2, secondary_y=True)

# --- Plot 5: Torque vs phi (Legend 5) ---
fig.add_trace(go.Scatter(x=phi_deg, y=T_pedal, mode='lines', name=name_T, line=dict(color='#9467bd', width=3), legend='legend5'), row=3, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=phi_deg, y=F_l, mode='lines', name=name_Fl, line=style_Fl, legend='legend5'), row=3, col=1, secondary_y=True)


# ==========================================
# 5. Achsen-Titel und Invertierungen
# ==========================================
# Plot 1
fig.update_xaxes(title_text="Sled Position x (mm)", row=1, col=1)
fig.update_yaxes(title_text="Force (N)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Loadcell Force (N)", row=1, col=1, secondary_y=True)

# Plot 2
fig.update_xaxes(title_text="Horizontal Pos. D_hor (mm)", row=1, col=2)
fig.update_yaxes(title_text="Force (N)", row=1, col=2, secondary_y=False)
fig.update_yaxes(title_text="Loadcell Force (N)", row=1, col=2, secondary_y=True)

# Plot 3
fig.update_xaxes(title_text="Pedal Angle φ (°)", autorange="reversed", row=2, col=1)
fig.update_yaxes(title_text="Force (N)", row=2, col=1, secondary_y=False)
fig.update_yaxes(title_text="Loadcell Force (N)", row=2, col=1, secondary_y=True)

# Plot 4
fig.update_xaxes(title_text="Horizontal Pos. D_hor (mm)", row=2, col=2)
fig.update_yaxes(title_text="Torque (Nm)", row=2, col=2, secondary_y=False)
fig.update_yaxes(title_text="Loadcell Force (N)", row=2, col=2, secondary_y=True)

# Plot 5
fig.update_xaxes(title_text="Pedal Angle φ (°)", autorange="reversed", row=3, col=1)
fig.update_yaxes(title_text="Torque (Nm)", row=3, col=1, secondary_y=False)
fig.update_yaxes(title_text="Loadcell Force (N)", row=3, col=1, secondary_y=True)


# ==========================================
# 6. Layout-Anpassungen & Mehrfache Legenden
# ==========================================
fig.update_layout(
    title_text="Comprehensive Pedal Kinematics & Dynamics",
    title_font_size=24,
    title_y=0.98,
    height=1250, # Etwas höher für die extra Legenden
    width=1300,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(t=160), # Viel Platz oben, damit Haupttitel nicht abgeschnitten wird

    # 5 individuelle Legenden, exakt über den jeweiligen Subplots positioniert
    legend=dict(x=0.0, y=1.01, xanchor='left', yanchor='bottom', orientation='h', font=dict(size=11), bgcolor='rgba(0,0,0,0)'),
    legend2=dict(x=0.55, y=1.01, xanchor='left', yanchor='bottom', orientation='h', font=dict(size=11), bgcolor='rgba(0,0,0,0)'),
    legend3=dict(x=0.0, y=0.625, xanchor='left', yanchor='bottom', orientation='h', font=dict(size=11), bgcolor='rgba(0,0,0,0)'),
    legend4=dict(x=0.55, y=0.625, xanchor='left', yanchor='bottom', orientation='h', font=dict(size=11), bgcolor='rgba(0,0,0,0)'),
    legend5=dict(x=0.0, y=0.24, xanchor='left', yanchor='bottom', orientation='h', font=dict(size=11), bgcolor='rgba(0,0,0,0)'),
)

# Wir schieben die Subplot-Titel etwas nach oben, damit sie nicht mit den Legenden überlappen
for annotation in fig['layout']['annotations']:
    annotation['y'] += 0.04

# Gitterlinien konfigurieren
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', secondary_y=False)
fig.update_yaxes(showgrid=False, secondary_y=True)

# Leeren Graphen unten rechts unsichtbar machen
fig.update_xaxes(visible=False, row=3, col=2)
fig.update_yaxes(visible=False, row=3, col=2, secondary_y=False)
fig.update_yaxes(visible=False, row=3, col=2, secondary_y=True)

fig.show()